# 07/08 — Contrastive Latent Alignment

Controlled V3 extension: same fixed split, architecture, seeds, evaluation and aggregation. Only the alignment objective differs. The class-level aggregate statistics may disclose information and are not formally private.


In [3]:
%pip install scikit-learn torch

import os, copy, random, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

SPLIT_SEED = 42
SEEDS = [42, 52, 62, 72, 82]
LATENT_DIM = 8
ALIGN_LAMBDA = 0.10  # predefined; do not tune on the test set
print("Seeds:", SEEDS, "| alignment lambda:", ALIGN_LAMBDA)


Note: you may need to restart the kernel to use updated packages.
Seeds: [42, 52, 62, 72, 82] | alignment lambda: 0.1


## 1. Load data and preserve the V2 split protocol


In [4]:
DATASET1_PATH = "../data/diabetes_prediction_dataset.csv"
DATASET2_PATH = "../data/diabetes_dataset.csv"
df1 = pd.read_csv(DATASET1_PATH)
df2 = pd.read_csv(DATASET2_PATH)
TARGET1, TARGET2 = "diabetes", "diagnosed_diabetes"

df2_model = df2.drop(columns=[c for c in ["diabetes_stage","diabetes_risk_score"] if c in df2.columns]).copy()
X1, y1 = df1.drop(columns=[TARGET1]), df1[TARGET1].astype(int)
X2, y2 = df2_model.drop(columns=[TARGET2]), df2_model[TARGET2].astype(int)

X1_full, X1_test, y1_full, y1_test = train_test_split(X1,y1,test_size=.20,random_state=SPLIT_SEED,stratify=y1)
X2_full, X2_test, y2_full, y2_test = train_test_split(X2,y2,test_size=.20,random_state=SPLIT_SEED,stratify=y2)
X1_train,X1_val,y1_train,y1_val = train_test_split(X1_full,y1_full,test_size=.125,random_state=SPLIT_SEED,stratify=y1_full)
X2_train,X2_val,y2_train,y2_val = train_test_split(X2_full,y2_full,test_size=.125,random_state=SPLIT_SEED,stratify=y2_full)
print("D1:",len(X1_train),len(X1_val),len(X1_test))
print("D2:",len(X2_train),len(X2_val),len(X2_test))


D1: 70000 10000 20000
D2: 70000 10000 20000


## 2. Client-specific preprocessing


In [5]:
def build_preprocessor(X):
    cat=X.select_dtypes(include=["object","category","bool"]).columns.tolist()
    num=[c for c in X.columns if c not in cat]
    return ColumnTransformer([
        ("num",Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())]),num),
        ("cat",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]),cat)
    ])

prep1,prep2=build_preprocessor(X1_train),build_preprocessor(X2_train)
A_train=prep1.fit_transform(X1_train).astype(np.float32); A_val=prep1.transform(X1_val).astype(np.float32); A_test=prep1.transform(X1_test).astype(np.float32)
B_train=prep2.fit_transform(X2_train).astype(np.float32); B_val=prep2.transform(X2_val).astype(np.float32); B_test=prep2.transform(X2_test).astype(np.float32)
yA_train=np.asarray(y1_train,dtype=np.int64); yB_train=np.asarray(y2_train,dtype=np.int64)
print("Input dims:",A_train.shape[1],B_train.shape[1])


Input dims: 15 46


## 3. Same V2 architecture and utilities


In [6]:
class ClientEncoder(nn.Module):
    def __init__(self,input_dim,latent_dim=LATENT_DIM):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(input_dim,32),nn.ReLU(),nn.Linear(32,latent_dim),nn.ReLU())
    def forward(self,x): return self.net(x)

class SharedPredictor(nn.Module):
    def __init__(self,latent_dim=LATENT_DIM):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(latent_dim,8),nn.ReLU(),nn.Linear(8,1))
    def forward(self,z): return self.net(z).squeeze(1)

class ClientModel(nn.Module):
    def __init__(self,encoder,predictor):
        super().__init__(); self.encoder=encoder; self.predictor=predictor
    def forward(self,x): return self.predictor(self.encoder(x))

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

def loader(X,y,batch=512):
    return DataLoader(TensorDataset(torch.tensor(X,dtype=torch.float32),torch.tensor(np.asarray(y),dtype=torch.float32)),batch_size=batch,shuffle=True)

def train_standard(model,X,y,epochs=5,lr=1e-3):
    opt=torch.optim.Adam(model.parameters(),lr=lr); loss_fn=nn.BCEWithLogitsLoss()
    model.train()
    for _ in range(epochs):
        for xb,yb in loader(X,y):
            opt.zero_grad(); loss=loss_fn(model(xb),yb); loss.backward(); opt.step()
    return model

def predict_prob(model,X):
    model.eval()
    with torch.no_grad(): return torch.sigmoid(model(torch.tensor(X,dtype=torch.float32))).cpu().numpy()

def best_threshold(y,p):
    ts=np.linspace(.05,.95,181)
    fs=[f1_score(y,(p>=t).astype(int),zero_division=0) for t in ts]
    return float(ts[int(np.argmax(fs))])

def get_metrics(y,p,t):
    pred=(p>=t).astype(int)
    return {"Threshold":t,"Accuracy":accuracy_score(y,pred),"Precision":precision_score(y,pred,zero_division=0),
            "Recall":recall_score(y,pred,zero_division=0),"F1":f1_score(y,pred,zero_division=0),"ROC-AUC":roc_auc_score(y,p)}

def average_predictors(ps,ns):
    total=float(sum(ns)); state=copy.deepcopy(ps[0].state_dict())
    for k in state:
        state[k]=torch.zeros_like(state[k])
        for p,n in zip(ps,ns): state[k]+=p.state_dict()[k]*(n/total)
    result=SharedPredictor(); result.load_state_dict(state); return result


## 4. Contrastive latent alignment
Compute detached class-level summaries before each federated round.


In [7]:
def prototypes(encoder, X, y):
    """Detached class means from one client's latent representations."""
    encoder.eval()
    with torch.no_grad():
        z = encoder(torch.as_tensor(X, dtype=torch.float32))
        yt = torch.as_tensor(np.asarray(y), dtype=torch.long)
        return {c: z[yt == c].mean(0).detach().clone() for c in (0, 1)}

def train_aligned(model,X,y,target,epochs=3,lr=1e-3,align_lambda=ALIGN_LAMBDA):
    opt=torch.optim.Adam(model.parameters(),lr=lr)
    bce=nn.BCEWithLogitsLoss()
    model.train()
    for _ in range(epochs):
        for xb,yb in loader(X,y):
            opt.zero_grad()
            z=model.encoder(xb); logits=model.predictor(z)
            # Supervised prototype contrast: correct class target must be closer
            # than the opposite class target (temperature-scaled softmax).
            targets=torch.stack([target[0],target[1]]).to(z.device)
            distances=torch.cdist(z,targets).square()
            contrastive=nn.functional.cross_entropy(-distances/0.5,yb.long())
            loss=bce(logits,yb)+align_lambda*contrastive
            loss.backward();opt.step()
    return model


## 5. One seed: unchanged local control vs aligned federation


In [8]:
def run_seed(seed,rounds=5,local_epochs=3):
    set_seed(seed)
    localA=ClientModel(ClientEncoder(A_train.shape[1]),SharedPredictor())
    localB=ClientModel(ClientEncoder(B_train.shape[1]),SharedPredictor())
    train_standard(localA,A_train,y1_train,epochs=20)
    train_standard(localB,B_train,y2_train,epochs=20)

    set_seed(seed)
    encA,encB=ClientEncoder(A_train.shape[1]),ClientEncoder(B_train.shape[1])
    global_pred=SharedPredictor()

    for _ in range(rounds):
        protoA,protoB=prototypes(encA,A_train,yA_train),prototypes(encB,B_train,yB_train)
        modelA=ClientModel(encA,copy.deepcopy(global_pred))
        modelB=ClientModel(encB,copy.deepcopy(global_pred))
        train_aligned(modelA,A_train,yA_train,protoB,epochs=local_epochs)
        train_aligned(modelB,B_train,yB_train,protoA,epochs=local_epochs)
        encA,encB=modelA.encoder,modelB.encoder
        global_pred=average_predictors([modelA.predictor,modelB.predictor],[len(A_train),len(B_train)])

    fedA=ClientModel(encA,copy.deepcopy(global_pred)); fedB=ClientModel(encB,copy.deepcopy(global_pred))
    cases=[
        ("Local","Dataset 1",localA,A_val,y1_val,A_test,y1_test),
        ("Local","Dataset 2",localB,B_val,y2_val,B_test,y2_test),
        ("Federated Contrastive","Dataset 1",fedA,A_val,y1_val,A_test,y1_test),
        ("Federated Contrastive","Dataset 2",fedB,B_val,y2_val,B_test,y2_test)]
    rows=[]
    for approach,dataset,model,Xv,yv,Xt,yt in cases:
        pv=predict_prob(model,Xv); t=best_threshold(yv,pv); pt=predict_prob(model,Xt)
        rows.append({"Seed":seed,"Approach":approach,"Dataset":dataset,**get_metrics(yt,pt,t)})
    return rows


## 6. Run the same five seeds


In [9]:
rows=[]
for i,s in enumerate(SEEDS,1):
    print(f"Running seed {s} ({i}/{len(SEEDS)})...")
    rows.extend(run_seed(s))
    print("Completed",s)
experiment_results=pd.DataFrame(rows)
metric_cols=["Accuracy","Precision","Recall","F1","ROC-AUC"]
display(experiment_results.round(4))


Running seed 42 (1/5)...
Completed 42
Running seed 52 (2/5)...
Completed 52
Running seed 62 (3/5)...
Completed 62
Running seed 72 (4/5)...
Completed 72
Running seed 82 (5/5)...
Completed 82


,Seed,Approach,Dataset,Threshold,Accuracy,Precision,Recall,F1,ROC-AUC
0,42,Local,Dataset 1,0.415,0.9698,0.9329,0.6953,0.7968,0.9758
1,42,Local,Dataset 2,0.355,0.9096,0.9784,0.8684,0.9201,0.9416
2,42,Federated Contrastive,Dataset 1,0.935,0.9584,0.8019,0.6788,0.7353,0.9617
3,42,Federated Contrastive,Dataset 2,0.160,0.9087,0.9861,0.8600,0.9187,0.9411
4,52,Local,Dataset 1,0.470,0.9684,0.9428,0.6694,0.7829,0.9747
5,52,Local,Dataset 2,0.455,0.9098,0.9879,0.8601,0.9196,0.9420
6,52,Federated Contrastive,Dataset 1,0.950,0.9672,0.8932,0.6982,0.7838,0.9676
7,52,Federated Contrastive,Dataset 2,0.160,0.9082,0.9859,0.8593,0.9183,0.9410
8,62,Local,Dataset 1,0.460,0.9700,0.9568,0.6776,0.7934,0.9732
9,62,Local,Dataset 2,0.570,0.9137,0.9936,0.8618,0.9230,0.9419


## 7. Mean ± standard deviation


In [10]:
summary=[]
for (dataset,approach),g in experiment_results.groupby(["Dataset","Approach"]):
    r={"Dataset":dataset,"Approach":approach}
    for m in metric_cols: r[m]=f"{g[m].mean():.4f} ± {g[m].std(ddof=1):.4f}"
    summary.append(r)
experiment_summary=pd.DataFrame(summary)
display(experiment_summary)


,Dataset,Approach,Accuracy,Precision,Recall,F1,ROC-AUC
0,Dataset 1,Federated Contrastive,0.9626 ± 0.0040,0.8409 ± 0.0586,0.6965 ± 0.0291,0.7603 ± 0.0179,0.9668 ± 0.0034
1,Dataset 1,Local,0.9688 ± 0.0020,0.9379 ± 0.0212,0.6775 ± 0.0122,0.7867 ± 0.0131,0.9736 ± 0.0018
2,Dataset 2,Federated Contrastive,0.9088 ± 0.0005,0.9888 ± 0.0032,0.8577 ± 0.0022,0.9186 ± 0.0003,0.9406 ± 0.0006
3,Dataset 2,Local,0.9112 ± 0.0017,0.9883 ± 0.0060,0.8623 ± 0.0035,0.9210 ± 0.0013,0.9417 ± 0.0003


## 8. Federated Contrastive − Local deltas


In [11]:
p=experiment_results.pivot(index=["Seed","Dataset"],columns="Approach",values=metric_cols)
d=[]
for s in SEEDS:
    for ds in ["Dataset 1","Dataset 2"]:
        r={"Seed":s,"Dataset":ds}
        for m in metric_cols: r["Delta "+m]=p.loc[(s,ds),(m,"Federated Contrastive")]-p.loc[(s,ds),(m,"Local")]
        d.append(r)
experiment_deltas=pd.DataFrame(d)
display(experiment_deltas.round(4))
display(experiment_deltas.groupby("Dataset")[[c for c in experiment_deltas if c.startswith("Delta")]].agg(["mean","std"]).round(4))


,Seed,Dataset,Delta Accuracy,Delta Precision,Delta Recall,Delta F1,Delta ROC-AUC
0,42,Dataset 1,-0.0114,-0.1310,-0.0165,-0.0615,-0.0141
1,42,Dataset 2,-0.0009,0.0076,-0.0084,-0.0014,-0.0005
2,52,Dataset 1,-0.0012,-0.0497,0.0288,0.0008,-0.0071
3,52,Dataset 2,-0.0015,-0.0020,-0.0008,-0.0013,-0.0010
4,62,Dataset 1,-0.0067,-0.1053,0.0106,-0.0322,-0.0051
5,62,Dataset 2,-0.0042,-0.0016,-0.0057,-0.0039,-0.0018
6,72,Dataset 1,-0.0069,-0.1424,0.0818,-0.0120,-0.0001
7,72,Dataset 2,-0.0027,-0.0016,-0.0030,-0.0024,-0.0004
8,82,Dataset 1,-0.0046,-0.0567,-0.0100,-0.0269,-0.0078
9,82,Dataset 2,-0.0030,0.0002,-0.0052,-0.0029,-0.0015


Delta Accuracy         Delta Precision         Delta Recall          \
                    mean     std            mean     std         mean     std   
Dataset                                                                         
Dataset 1        -0.0062  0.0037         -0.0970  0.0423       0.0189  0.0394   
Dataset 2        -0.0024  0.0013          0.0005  0.0041      -0.0046  0.0029   

          Delta F1         Delta ROC-AUC          
              mean     std          mean     std  
Dataset                                           
Dataset 1  -0.0264  0.0235       -0.0068  0.0051  
Dataset 2  -0.0024  0.0011       -0.0010  0.0006

## 9. Direct V3 vs V2 comparison


In [12]:
V2_PATH="../models/prototype_v2_multiseed_results.csv"
if os.path.exists(V2_PATH):
    v2=pd.read_csv(V2_PATH)
    v2f=v2[v2.Approach=="Federated"][["Seed","Dataset"]+metric_cols].rename(columns={m:"V2_Fed_"+m for m in metric_cols})
    v3f=experiment_results[experiment_results.Approach=="Federated Contrastive"][["Seed","Dataset"]+metric_cols].rename(columns={m:"contrastive_Fed_"+m for m in metric_cols})
    v2_comparison=v2f.merge(v3f,on=["Seed","Dataset"])
    for m in metric_cols: v2_comparison["contrastive_minus_V2_"+m]=v2_comparison["contrastive_Fed_"+m]-v2_comparison["V2_Fed_"+m]
    display(v2_comparison.round(4))
    cols=["contrastive_minus_V2_"+m for m in metric_cols]
    display(v2_comparison.groupby("Dataset")[cols].agg(["mean","std"]).round(4))
else:
    v2_comparison=None
    print("V2 CSV not found:",V2_PATH)


,Seed,Dataset,V2_Fed_Accuracy,V2_Fed_Precision,V2_Fed_Recall,V2_Fed_F1,V2_Fed_ROC-AUC,contrastive_Fed_Accuracy,contrastive_Fed_Precision,contrastive_Fed_Recall,contrastive_Fed_F1,contrastive_Fed_ROC-AUC,contrastive_minus_V2_Accuracy,contrastive_minus_V2_Precision,contrastive_minus_V2_Recall,contrastive_minus_V2_F1,contrastive_minus_V2_ROC-AUC
0,42,Dataset 1,0.9678,0.9483,0.6576,0.7767,0.9636,0.9584,0.8019,0.6788,0.7353,0.9617,-0.0094,-0.1463,0.0212,-0.0414,-0.0019
1,42,Dataset 2,0.9114,0.9885,0.8622,0.9211,0.9421,0.9087,0.9861,0.8600,0.9187,0.9411,-0.0027,-0.0025,-0.0022,-0.0024,-0.0010
2,52,Dataset 1,0.9703,0.9784,0.6653,0.7920,0.9709,0.9672,0.8932,0.6982,0.7838,0.9676,-0.0030,-0.0852,0.0329,-0.0083,-0.0033
3,52,Dataset 2,0.9113,0.9895,0.8613,0.9210,0.9415,0.9082,0.9859,0.8593,0.9183,0.9410,-0.0030,-0.0035,-0.0020,-0.0027,-0.0006
4,62,Dataset 1,0.9620,0.8666,0.6535,0.7451,0.9673,0.9633,0.8515,0.6882,0.7612,0.9681,0.0013,-0.0151,0.0347,0.0161,0.0008
5,62,Dataset 2,0.9122,0.9849,0.8669,0.9221,0.9419,0.9095,0.9920,0.8561,0.9190,0.9401,-0.0027,0.0071,-0.0108,-0.0031,-0.0019
6,72,Dataset 1,0.9703,0.9431,0.6924,0.7985,0.9713,0.9585,0.7614,0.7453,0.7533,0.9710,-0.0118,-0.1817,0.0529,-0.0452,-0.0003
7,72,Dataset 2,0.9122,0.9900,0.8622,0.9217,0.9415,0.9086,0.9876,0.8585,0.9186,0.9411,-0.0035,-0.0024,-0.0037,-0.0032,-0.0005
8,82,Dataset 1,0.9712,0.9707,0.6812,0.8006,0.9724,0.9655,0.8964,0.6718,0.7680,0.9656,-0.0056,-0.0743,-0.0094,-0.0326,-0.0068
9,82,Dataset 2,0.9130,0.9930,0.8610,0.9223,0.9417,0.9090,0.9925,0.8548,0.9185,0.9399,-0.0040,-0.0005,-0.0062,-0.0038,-0.0018


contrastive_minus_V2_Accuracy          \
                                   mean     std   
Dataset                                           
Dataset 1                       -0.0057  0.0052   
Dataset 2                       -0.0032  0.0006   

          contrastive_minus_V2_Precision         contrastive_minus_V2_Recall  \
                                    mean     std                        mean   
Dataset                                                                        
Dataset 1                        -0.1005  0.0650                      0.0265   
Dataset 2                        -0.0004  0.0043                     -0.0050   

                  contrastive_minus_V2_F1          \
              std                    mean     std   
Dataset                                             
Dataset 1  0.0231                 -0.0223  0.0258   
Dataset 2  0.0037                 -0.0030  0.0006   

          contrastive_minus_V2_ROC-AUC          
                                  mean     std  
Dataset                                         
Dataset 1                      -0.0023  0.0030  
Dataset 2                      -0.0011  0.0007

## 10. Save V3 outputs


In [13]:
os.makedirs("../models",exist_ok=True)
experiment_results.to_csv("../models/prototype_contrastive_multiseed_results.csv",index=False)
experiment_summary.to_csv("../models/prototype_contrastive_multiseed_summary.csv",index=False)
experiment_deltas.to_csv("../models/prototype_contrastive_multiseed_deltas.csv",index=False)
if v2_comparison is not None: v2_comparison.to_csv("../models/prototype_contrastive_vs_v2.csv",index=False)
print("Saved experiment outputs.")


Saved experiment outputs.
